# NIFTY-50 Investment Intelligence: Exploration & Model Training
This notebook provides an interactive walkthrough of the **NIFTY-50 Investment Intelligence Platform** pipeline, including:
1. **Data Loading & Cleaning**: Accessing raw stock prices and metadata.
2. **Exploratory Data Analysis (EDA)**: Visualizing distributions, correlations, and price trajectories.
3. **Feature Engineering**: Computing technical indicators (RSI, MACD, Bollinger Bands, Volatility, Momentum).
4. **Model Training & Validation**: Training and evaluating Machine Learning models for return forecasting and direction prediction.
5. **Portfolio Construction**: Running Mean-Variance Optimization (MVO) for Conservative, Balanced, and Aggressive profiles.
6. **Explainability & Anomaly Detection**: Feature importance analysis and identifying market anomalies.

## 1. Setup and Environment

In [ ]:
# Install dependencies if running in Google Colab
# !pip install pandas numpy scikit-learn plotly scipy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from src.data_loader import NiftyDataLoader
from src.features import compute_technical_indicators, generate_predictor_features
from src.predictor import StockPredictorEngine
from src.portfolio import PortfolioConstructor
from src.risk import RiskAssessor
from src.anomaly import MarketAnomalyDetector
from src.explainability import ExplainabilityEngine

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries successfully imported!")

## 2. Data Loading & Cleaning
We load the stock metadata and historical daily prices using `NiftyDataLoader`. If no files exist in the `data/` directory, it automatically generates a realistic synthetic dataset for you.

In [ ]:
loader = NiftyDataLoader()
metadata = loader.load_metadata()
print("Stock Metadata:")
display(metadata)

stock_data = loader.load_all_stocks()
print(f"\nSuccessfully loaded data for {len(stock_data)} stocks.")

## 3. Exploratory Data Analysis (EDA)
Let's look at the price chart of a selected stock (e.g. HDFCBANK) and plot the correlation/covariance heatmap of returns.

In [ ]:
symbol = "HDFCBANK"
df_stock = stock_data[symbol]

plt.figure(figsize=(12, 5))
plt.plot(df_stock['Date'], df_stock['Close'], label=symbol, color='#0284c7')
plt.title(f"{symbol} Historical Close Price")
plt.xlabel("Date")
plt.ylabel("Price (INR)")
plt.legend()
plt.show()

In [ ]:
# Calculate returns and show the correlation matrix
returns_dict = {}
for sym, df in stock_data.items():
    returns_dict[sym] = df['Close'].pct_change().fillna(0)
returns_df = pd.DataFrame(returns_dict)

plt.figure(figsize=(10, 8))
sns.heatmap(returns_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of NIFTY-50 Stocks")
plt.show()

## 4. Feature Engineering
We compute technical indicators such as RSI, MACD, and Bollinger Bands using our vectorized operations in `src/features.py`.

In [ ]:
df_indicators = compute_technical_indicators(df_stock)
print(f"Generated indicators for {symbol}. First 5 rows:")
display(df_indicators[['Date', 'Close', 'SMA_50', 'RSI_14', 'MACD', 'BB_Upper', 'BB_Lower']].head())

In [ ]:
# Plot Bollinger Bands & RSI
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(df_indicators['Date'], df_indicators['Close'], label='Close', color='black')
ax1.plot(df_indicators['Date'], df_indicators['BB_Upper'], label='BB Upper', color='green', linestyle='--')
ax1.plot(df_indicators['Date'], df_indicators['BB_Lower'], label='BB Lower', color='red', linestyle='--')
ax1.fill_between(df_indicators['Date'], df_indicators['BB_Lower'], df_indicators['BB_Upper'], color='gray', alpha=0.1)
ax1.set_title(f"{symbol} Bollinger Bands")
ax1.legend()

ax2.plot(df_indicators['Date'], df_indicators['RSI_14'], label='RSI (14)', color='purple')
ax2.axhline(70, color='red', linestyle='--', alpha=0.5)
ax2.axhline(30, color='green', linestyle='--', alpha=0.5)
ax2.set_title("Relative Strength Index (RSI)")
ax2.set_ylim(0, 100)
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Model Training and Predictions
We train models to forecast the 5-day forward return (regression) and direction (classification) using the `StockPredictorEngine`.

In [ ]:
predictor_engine = StockPredictorEngine()
reg_metrics, clf_metrics = predictor_engine.train_models(df_stock, symbol)

print("Regressor Metrics:")
for k, v in reg_metrics.items():
    print(f"  {k}: {v:.6f}")
    
print("\nClassifier Metrics:")
for k, v in clf_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Generate future predictions on latest data
predictions = predictor_engine.predict(df_stock, symbol)
print(f"Predictions for {symbol} on {predictions['Date']}:")
for k, v in predictions.items():
    print(f"  {k}: {v}")

## 6. Portfolio Construction & Optimization
We build portfolios for different risk tolerances using the `PortfolioConstructor` class, applying Mean-Variance Optimization (MVO).

In [ ]:
risk_free_rate = 0.0675  # 6.75% risk-free rate
p_constructor = PortfolioConstructor(stock_data, metadata)

for profile in ["Conservative", "Balanced", "Aggressive"]:
    p = p_constructor.construct_portfolio(profile, risk_free_rate=risk_free_rate)
    print("=" * 60)
    print(f"PROFILE: {profile}")
    print(f"  Expected Return: {p['Expected_Annualized_Return']*100:.2f}%")
    print(f"  Volatility: {p['Expected_Annualized_Volatility']*100:.2f}%")
    print(f"  Sharpe Ratio: {p['Sharpe_Ratio']:.4f}")
    print("  Top Holdings:")
    display(p["Allocations"].head(3))
    print(f"  Justification: {p['Justification']}\n")

## 7. Explainable AI & Anomaly Detection
Let's look at the permutation importance of features in our models and check for recent anomalies.

In [ ]:
reg_model, feat_cols, _ = predictor_engine.load_model(symbol, "regressor")
explainer = ExplainabilityEngine()
importance_df = explainer.compute_model_feature_importance(reg_model, df_stock, feat_cols)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(10), x="Importance_Mean", y="Feature", palette="viridis")
plt.title(f"Feature Importance for {symbol} Return Regressor")
plt.xlabel("Permutation Importance (Loss Decrease)")
plt.ylabel("Feature")
plt.show()

In [ ]:
# Detect market anomalies in HDFCBANK
detector = MarketAnomalyDetector()
anomalies = detector.detect_anomalies(df_stock)
print(f"Total anomalies detected in {symbol}: {len(anomalies)}")
if not anomalies.empty:
    display(anomalies.head(10))